# Experiment 06: EE-TransUNet with ViT-Tiny

**Date:** 2025-01-08  
**Author:** Sebastian Siedler  
**Objective:** Implement Vision Transformer architecture for optic disc/cup segmentation

---

## 📋 Experiment Overview

### Hypothesis
Vision Transformers can capture global context better than CNNs through self-attention mechanisms, potentially improving segmentation of circular structures (optic disc/cup).

### Key Changes from Previous Experiments
1. **Architecture:** Switched from CNN-based U-Net to Vision Transformer (EE-TransUNet)
2. **Attention Mechanism:** Introduced multi-head self-attention for global context
3. **Model Size:** Custom ViT-Tiny variant (5.7M parameters vs 24.5M in ResNet34-UNet)

### Configuration
- **Model:** EE-TransUNet ViT-Tiny
- **Preprocessing:** None (no CLAHE)
- **Dataset:** REFUGE (cropped masks)
- **Training samples:** 400
- **Validation samples:** 80
- **Test samples:** 400

### Hyperparameters
```python
{
    'epochs': 39,  # Stopped early due to instability
    'batch_size': 16,
    'learning_rate': 0.01,  # High LR caused validation spikes
    'optimizer': 'SGD',
    'img_size': 224,
    'patch_size': 16,
    'hidden_size': 256,
    'num_heads': 4,
    'num_layers': 6
}
```

**Note:** This notebook contains the training workflow. The actual training was already completed and stopped at epoch 39. The trained model is in `results/checkpoint_epoch_39.pth`.

---

## 1️⃣ Environment Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import json
from datetime import datetime

# Project imports
from models.ee_transunet import VisionTransformer
from models.configs import get_tiny_config
from data_loader.dataset import RetinaDataset, RetinaDatasetTest
from utils.metrics import batch_metrics, calculate_cdr

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2️⃣ Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    # Model
    'model': 'EE-TransUNet ViT-Tiny',
    'model_name': 'ViT-Tiny',
    'img_size': 224,
    'patch_size': 16,
    'n_classes': 2,
    'n_skip': 0,  # No skip connections
    
    # Data
    'data_dir': '../../datasets/REFUGE',
    'train_csv': '../../datasets/REFUGE/REFUGETrain.csv',
    'val_csv': '../../datasets/REFUGE/REFUGE1Val.csv',
    'test_csv': '../../datasets/REFUGE/REFUGE1Test.csv',
    'use_cropped': True,
    'use_clahe': False,
    
    # Training
    'epochs': 50,
    'batch_size': 16,
    'learning_rate': 0.01,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'num_workers': 4,
    
    # Output
    'output_dir': './results',
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(f"{CONFIG['output_dir']}/visualizations", exist_ok=True)

# Save configuration
with open(f"{CONFIG['output_dir']}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=4)

print("✅ Configuration saved")
print(json.dumps(CONFIG, indent=2))

## 3️⃣ Data Loading

In [ ]:
# Create datasets
print("Loading datasets...")

train_dataset = RetinaDataset(
    csv_file=CONFIG['train_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

val_dataset = RetinaDataset(
    csv_file=CONFIG['val_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

test_dataset = RetinaDatasetTest(
    csv_file=CONFIG['test_csv'],
    root_dir=CONFIG['data_dir'],
    use_cropped=CONFIG['use_cropped'],
    use_clahe=CONFIG['use_clahe'],
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
                         num_workers=CONFIG['num_workers'], pin_memory=True)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Validation samples: {len(val_dataset)}")
print(f"✅ Test samples: {len(test_dataset)}")

## 4️⃣ Model Definition

In [ ]:
# Load model configuration
config_vit = get_tiny_config()
config_vit.n_classes = CONFIG['n_classes']
config_vit.n_skip = CONFIG['n_skip']

# Create model
model = VisionTransformer(config_vit, img_size=CONFIG['img_size'], num_classes=CONFIG['n_classes']).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ EE-TransUNet ViT-Tiny model created")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model config:")
print(f"      - Hidden size: {config_vit.hidden_size}")
print(f"      - MLP dim: {config_vit.transformer.mlp_dim}")
print(f"      - Num heads: {config_vit.transformer.num_heads}")
print(f"      - Num layers: {config_vit.transformer.num_layers}")
print(f"      - Patch size: {config_vit.patches.size}")

In [ ]:
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=CONFIG['learning_rate'], 
                      momentum=CONFIG['momentum'], weight_decay=CONFIG['weight_decay'])

print("✅ Loss function: BCEWithLogitsLoss")
print("✅ Optimizer: SGD with momentum")

## 5️⃣ Training

**Note:** Training was already completed and stopped at epoch 39. The code below shows the training workflow used.

In [ ]:
# Training functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0
    epoch_dice = 0
    
    pbar = tqdm(loader, desc='Training')
    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device)
        masks = masks.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            metrics = batch_metrics(outputs, masks)
            epoch_loss += loss.item()
            epoch_dice += metrics['dice_mean']
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'dice': f"{metrics['dice_mean']:.4f}"})
    
    return epoch_loss / len(loader), epoch_dice / len(loader)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    epoch_dice_disc = 0
    epoch_dice_cup = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            metrics = batch_metrics(outputs, masks)
            epoch_loss += loss.item()
            epoch_dice += metrics['dice_mean']
            epoch_dice_disc += metrics['dice_disc']
            epoch_dice_cup += metrics['dice_cup']
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'dice': f"{metrics['dice_mean']:.4f}"})
    
    return {
        'loss': epoch_loss / len(loader),
        'dice_mean': epoch_dice / len(loader),
        'dice_disc': epoch_dice_disc / len(loader),
        'dice_cup': epoch_dice_cup / len(loader),
    }

print("✅ Training functions defined")

In [ ]:
# Training loop (ALREADY COMPLETED - this is reference code)
# The actual training was done with src/main.py and stopped at epoch 39

# Uncomment below to retrain from scratch
"""
history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [],
           'val_dice_disc': [], 'val_dice_cup': []}
best_dice = 0.0
start_time = datetime.now()

print(f"\n{'='*60}")
print("Starting training...")
print(f"{'='*60}\n")

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 60)
    
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_dice'].append(train_dice)
    history['val_loss'].append(val_metrics['loss'])
    history['val_dice'].append(val_metrics['dice_mean'])
    history['val_dice_disc'].append(val_metrics['dice_disc'])
    history['val_dice_cup'].append(val_metrics['dice_cup'])
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
    print(f"  Val Loss:   {val_metrics['loss']:.4f} | Val Dice:   {val_metrics['dice_mean']:.4f}")
    print(f"  Val Disc:   {val_metrics['dice_disc']:.4f} | Val Cup:    {val_metrics['dice_cup']:.4f}")
    
    if val_metrics['dice_mean'] > best_dice:
        best_dice = val_metrics['dice_mean']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
            'args': CONFIG,
        }, f"{CONFIG['output_dir']}/best_model.pth")
        print(f"  ✅ Best model saved (Dice: {best_dice:.4f})")

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds() / 3600

print(f"\n{'='*60}")
print(f"Training completed in {training_time:.2f} hours")
print(f"Best validation Dice: {best_dice:.4f}")
print(f"{'='*60}")
"""

print("⚠️ Training was already completed externally (stopped at epoch 39)")
print("⚠️ Trained model available at: results/checkpoint_epoch_39.pth")
print("⚠️ Uncomment the code above to retrain from scratch")

### 📈 Training History (From Completed Training)

The model was trained for 39 epochs with the following progression:

```
Epoch 9:  Train Dice: 0.85, Val Dice: 0.82, Val Loss: 0.12
Epoch 19: Train Dice: 0.87, Val Dice: 0.84, Val Loss: 0.11
Epoch 29: Train Dice: 0.88, Val Dice: 0.85, Val Loss: 0.44 (spike!)
Epoch 39: Train Dice: 0.88, Val Dice: 0.82, Val Loss: 0.12
```

**Issues Observed:**
- High learning rate (0.01) caused validation loss spikes
- Overfitting gap (Train 0.88 vs Val 0.82)
- No data augmentation contributed to overfitting

## 6️⃣ Testing

In [ ]:
# Load trained model checkpoint
print("Loading trained model from epoch 39...")
checkpoint = torch.load(f"{CONFIG['output_dir']}/checkpoint_epoch_39.pth", weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Model loaded successfully")
print(f"   Training epoch: {checkpoint['epoch']}")

In [ ]:
# Test model
print("\nTesting model on test set...")

test_metrics = {'dice_mean': [], 'dice_disc': [], 'dice_cup': [], 'cdr_mae': []}
all_predictions = []
all_masks = []
all_images = []

with torch.no_grad():
    pbar = tqdm(test_loader, desc='Testing')
    for images, masks in pbar:
        images_device = images.to(device)
        masks_device = masks.to(device)
        
        outputs = model(images_device)
        metrics = batch_metrics(outputs, masks_device)
        
        test_metrics['dice_mean'].append(metrics['dice_mean'])
        test_metrics['dice_disc'].append(metrics['dice_disc'])
        test_metrics['dice_cup'].append(metrics['dice_cup'])
        test_metrics['cdr_mae'].append(metrics['cdr_mae'])
        
        preds = torch.sigmoid(outputs).cpu()
        all_predictions.append(preds)
        all_masks.append(masks.cpu())
        all_images.append(images.cpu())
        
        pbar.set_postfix({'dice': f"{metrics['dice_mean']:.4f}", 'cdr_mae': f"{metrics['cdr_mae']:.4f}"})

# Calculate overall test results
test_results = {
    'dice_mean': np.mean(test_metrics['dice_mean']),
    'dice_std': np.std(test_metrics['dice_mean']),
    'dice_disc': np.mean(test_metrics['dice_disc']),
    'dice_cup': np.mean(test_metrics['dice_cup']),
    'cdr_mae': np.mean(test_metrics['cdr_mae']),
    'cdr_std': np.std(test_metrics['cdr_mae']),
}

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"Overall Dice Score:     {test_results['dice_mean']:.4f} ± {test_results['dice_std']:.4f} ({test_results['dice_mean']*100:.2f}%)")
print(f"Disc Dice Score:        {test_results['dice_disc']:.4f} ({test_results['dice_disc']*100:.2f}%)")
print(f"Cup Dice Score:         {test_results['dice_cup']:.4f} ({test_results['dice_cup']*100:.2f}%)")
print(f"CDR Mean Absolute Error: {test_results['cdr_mae']:.4f} ± {test_results['cdr_std']:.4f}")
print("="*60)

with open(f"{CONFIG['output_dir']}/test_results.json", 'w') as f:
    json.dump(test_results, f, indent=4)

print("\n✅ Test results saved to results/test_results.json")

## 7️⃣ Visualizations

In [ ]:
# Concatenate predictions
all_predictions = torch.cat(all_predictions, dim=0)
all_masks = torch.cat(all_masks, dim=0)
all_images = torch.cat(all_images, dim=0)

# Select samples for visualization (including best and worst)
dice_scores = []
for i in range(len(all_predictions)):
    disc_pred = (all_predictions[i, 0].numpy() > 0.5).astype(float)
    cup_pred = (all_predictions[i, 1].numpy() > 0.5).astype(float)
    disc_gt = all_masks[i, 0].numpy()
    cup_gt = all_masks[i, 1].numpy()
    
    disc_dice = 2 * (disc_pred * disc_gt).sum() / (disc_pred.sum() + disc_gt.sum() + 1e-8)
    cup_dice = 2 * (cup_pred * cup_gt).sum() / (cup_pred.sum() + cup_gt.sum() + 1e-8)
    overall_dice = (disc_dice + cup_dice) / 2
    dice_scores.append(overall_dice)

dice_scores = np.array(dice_scores)
best_idx = np.argmax(dice_scores)
worst_idx = np.argmin(dice_scores)
median_indices = np.argsort(dice_scores)[len(dice_scores)//2-1:len(dice_scores)//2+1]

vis_indices = [best_idx, median_indices[0], median_indices[1], worst_idx]

print(f"Selected samples: Best ({dice_scores[best_idx]:.3f}), Median, Median, Worst ({dice_scores[worst_idx]:.3f})")

In [ ]:
# Create visualizations
fig, axes = plt.subplots(4, 5, figsize=(20, 16))

for i, idx in enumerate(vis_indices):
    image = all_images[idx]
    mask_gt = all_masks[idx]
    pred = all_predictions[idx]
    
    # Denormalize image
    img_np = image.numpy().transpose(1, 2, 0)
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    
    disc_gt = mask_gt[0].numpy()
    cup_gt = mask_gt[1].numpy()
    disc_pred = (pred[0].numpy() > 0.5).astype(float)
    cup_pred = (pred[1].numpy() > 0.5).astype(float)
    
    disc_dice = 2 * (disc_pred * disc_gt).sum() / (disc_pred.sum() + disc_gt.sum() + 1e-8)
    cup_dice = 2 * (cup_pred * cup_gt).sum() / (cup_pred.sum() + cup_gt.sum() + 1e-8)
    overall_dice = (disc_dice + cup_dice) / 2
    
    cdr_gt = cup_gt.sum() / (disc_gt.sum() + 1e-8)
    cdr_pred = cup_pred.sum() / (disc_pred.sum() + 1e-8)
    
    sample_label = ['Best', 'Median', 'Median', 'Worst'][i]
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'{sample_label} Sample {idx}\nOverall Dice: {overall_dice:.3f}', fontweight='bold')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(disc_gt, cmap='gray')
    axes[i, 1].set_title(f'GT Disc\n(CDR: {cdr_gt:.3f})')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(cup_gt, cmap='gray')
    axes[i, 2].set_title('GT Cup')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(disc_pred, cmap='gray')
    axes[i, 3].set_title(f'Pred Disc\nDice: {disc_dice:.3f}')
    axes[i, 3].axis('off')
    
    axes[i, 4].imshow(cup_pred, cmap='gray')
    axes[i, 4].set_title(f'Pred Cup\nDice: {cup_dice:.3f}\nCDR: {cdr_pred:.3f}')
    axes[i, 4].axis('off')

plt.suptitle('EE-TransUNet ViT-Tiny - Test Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/visualizations/predictions.png", dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualizations saved to results/visualizations/predictions.png")

## 8️⃣ Results Summary

### Key Findings

**Performance Metrics:**
- **Overall Dice Score:** 87.21% ± 5.05%
- **Optic Disc Dice:** 93.97% (Excellent!)
- **Optic Cup Dice:** 80.45% (Good)
- **CDR MAE:** 0.0869 ± 0.0531 (Very accurate)

**Training Details:**
- **Training epochs:** 39 (stopped early)
- **Model parameters:** 5,674,114 (5.7M)
- **Training time:** ~2 hours
- **Device:** CPU (no GPU available)

### Comparison with Previous Experiments

| Experiment | Model | Params | Test Dice | Disc Dice | Cup Dice | CDR MAE |
|------------|-------|--------|-----------|-----------|----------|---------|
| 01 Baseline | UNet | 31.0M | 83.6% | 92.8% | 74.4% | 0.095 |
| 02 Small CLAHE | Small UNet | 7.7M | 86.5% | 93.5% | 79.5% | 0.089 |
| 03 ResNet34 | ResNet34-UNet | 24.5M | 72.1% | 87.3% | 56.9% | 0.156 |
| **06 ViT-Tiny** | **EE-TransUNet** | **5.7M** | **87.2%** | **94.0%** | **80.5%** | **0.087** |

### Observations

1. **Best Performance:** ViT-Tiny achieves the highest Dice score (87.2%) with the smallest model (5.7M params)
2. **Excellent Disc Segmentation:** 94.0% Disc Dice is the highest across all experiments
3. **Efficient:** Best parameter efficiency at 15.3 Dice points per million parameters
4. **Training Instability:** High LR (0.01) caused validation loss spikes - should use 0.001
5. **Overfitting:** No augmentation led to Train (88%) vs Val (82%) gap

### Next Steps

Based on these results, recommended improvements:
1. **Add Data Augmentation:** Rotations, flips, elastic deformations to reduce overfitting
2. **Reduce Learning Rate:** Use 0.001 with cosine annealing scheduler
3. **Add Early Stopping:** Monitor validation Dice, stop after 10 epochs without improvement
4. **Hybrid Architecture:** Try adding skip connections from encoder (n_skip > 0)
5. **Ensemble:** Combine ViT-Tiny predictions with Small UNet for potential boost

---

**Experiment Status:** ✅ Complete

**Conclusion:** EE-TransUNet ViT-Tiny demonstrates that Vision Transformers can achieve excellent performance for optic disc/cup segmentation with fewer parameters than CNN-based architectures. The model benefits from its global attention mechanism for capturing circular structures. With proper hyperparameter tuning and augmentation, this architecture shows great promise for medical image segmentation tasks.